In [195]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split ,RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LinearRegression , Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor , AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score ,mean_absolute_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from catboost import CatBoostRegressor
#from xgboost import XGBRegressor


In [196]:
df = pd.read_csv('data/student_performance.csv')

In [197]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [198]:
df.rename(columns={'race/ethnicity': 'race_ethnicity'}, inplace=True)

In [199]:
X = df.drop(['math score'], axis=1)
y = df['math score']

In [200]:
df['gender'].unique()

array(['female', 'male'], dtype=object)

In [201]:
X.shape , y.shape

((1000, 7), (1000,))

In [202]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [203]:
num_cols

Index(['reading score', 'writing score'], dtype='object')

In [204]:
cat_cols

Index(['gender', 'race_ethnicity', 'parental level of education', 'lunch',
       'test preparation course'],
      dtype='object')

In [205]:

num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns
print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# final_pipline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', LinearRegression())
# ])

# final_pipline.fit(X_train, y_train)
# y_pred = final_pipline.predict(X_test)

Numerical Columns: Index(['reading score', 'writing score'], dtype='object')
Categorical Columns: Index(['gender', 'race_ethnicity', 'parental level of education', 'lunch',
       'test preparation course'],
      dtype='object')


In [ ]:

def evaluate_model(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    return mae, mse, rmse, r2
   

In [207]:
X_train

,gender,race_ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
29,female,group D,master's degree,standard,none,70,75
535,female,group C,bachelor's degree,free/reduced,completed,83,83
695,female,group D,some college,free/reduced,none,89,86
557,male,group C,master's degree,free/reduced,none,67,66
836,male,group E,high school,standard,none,64,57
...,...,...,...,...,...,...,...
106,female,group D,master's degree,standard,none,100,100
270,male,group C,bachelor's degree,standard,none,63,61
860,female,group C,associate's degree,standard,none,62,53
435,male,group C,some college,free/reduced,completed,48,53


In [208]:
# mae, mse , r2 , rmse  = evaluate_model(y_test, y_pred)
# print(f'Mean Absolute Error: {mae}')
# print(f'Mean Squared Error: {mse}')
# print(f'R^2 Score: {r2}')
# print(f'Root Mean Squared Error: {rmse}')

In [209]:
X_test

,gender,race_ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
521,female,group C,associate's degree,standard,none,86,84
737,female,group B,some college,free/reduced,completed,66,73
740,male,group D,bachelor's degree,standard,none,73,72
660,male,group C,some college,free/reduced,none,77,73
411,male,group E,some college,standard,completed,83,78
...,...,...,...,...,...,...,...
408,female,group D,high school,free/reduced,completed,57,56
332,male,group E,associate's degree,standard,completed,56,53
208,female,group B,some college,free/reduced,none,81,76
613,female,group C,associate's degree,standard,none,77,74


In [210]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'KNN Regressor': KNeighborsRegressor(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'AdaBoost Regressor': AdaBoostRegressor(),
    'Support Vector Regressor': SVR(),
    'CatBoost Regressor': CatBoostRegressor(verbose=0)
    
}

results_train = {}
results_test = {}
for name , model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    train_mae, train_mse , train_r2 , train_rmse  = evaluate_model(y_train, y_train_pred)
    test_mae, test_mse , test_r2 , test_rmse  = evaluate_model(y_test, y_test_pred)
    results_train[name] = {'TRAIN_MAE': train_mae, 'TRAIN_MSE': train_mse, 'TRAIN_RMSE': train_rmse, 'TRAIN_R2': train_r2}
    results_test[name] = {'TEST_MAE': test_mae, 'TEST_MSE': test_mse, 'TEST_RMSE': test_rmse, 'TEST_R2': test_r2}
results_train_df = pd.DataFrame(results_train).T
results_test_df = pd.DataFrame(results_test).T

results_train_df.sort_values(by='TRAIN_R2', ascending=False)[['TRAIN_R2']]



,TRAIN_R2
Support Vector Regressor,6.740130
Lasso Regression,6.592500
AdaBoost Regressor,5.872142
KNN Regressor,5.582105
Ridge Regression,5.323498
Linear Regression,5.323051
CatBoost Regressor,3.095079
Random Forest Regressor,2.306313
Decision Tree Regressor,0.279508


In [211]:
results_test_df.sort_values(by='TEST_R2', ascending=False)[['TEST_R2']]

,TEST_R2
Support Vector Regressor,8.349003
Decision Tree Regressor,7.912332
KNN Regressor,7.392794
Lasso Regression,6.517328
AdaBoost Regressor,6.224351
CatBoost Regressor,6.036292
Random Forest Regressor,6.007094
Linear Regression,5.393994
Ridge Regression,5.393615
